# 01. 사전학습 모델 바로 쓰기 — pipeline

Hugging Face의 `pipeline` 은 사전학습 모델을 **단 몇 줄로** 사용하게 해줍니다.
토크나이징·추론·후처리를 자동으로 묶어주므로, 학습 없이 즉시 NLP 작업을 수행할 수 있습니다.

이 노트북에서 다루는 작업:
1. 감정 분석 (sentiment-analysis)
2. 개체명 인식 (NER)
3. 질의응답 (question-answering)
4. 요약 (summarization)
5. 제로샷 분류 (zero-shot-classification)

> 모델은 처음 실행 시 Hugging Face Hub에서 자동 다운로드되어 `/workspace/.cache/huggingface`에 저장됩니다.

In [ ]:
from transformers import pipeline
import torch

print('GPU 사용 가능:', torch.cuda.is_available())
device = 0 if torch.cuda.is_available() else -1

## 1. 감정 분석

문장이 긍정인지 부정인지 분류합니다.

In [ ]:
sentiment = pipeline('sentiment-analysis', device=device)
results = sentiment([
    'This movie was absolutely fantastic, I loved every minute!',
    'The plot was boring and the acting was terrible.',
])
for r in results:
    print(f"{r['label']:8s} (확신도 {r['score']:.3f})")

## 2. 개체명 인식 (NER)

문장에서 인명·지명·조직 등 개체를 찾아냅니다.

In [ ]:
ner = pipeline('ner', grouped_entities=True, device=device)
text = 'Hugging Face is a company based in New York and Paris.'
for ent in ner(text):
    print(f"{ent['word']:20s} -> {ent['entity_group']} ({ent['score']:.2f})")

## 3. 질의응답

주어진 지문(context)에서 질문의 답을 찾아 추출합니다.

In [ ]:
qa = pipeline('question-answering', device=device)
context = '''The Transformer architecture was introduced in 2017 in the paper
"Attention Is All You Need". It relies entirely on attention mechanisms,
dispensing with recurrence and convolutions entirely.'''
answer = qa(question='When was the Transformer introduced?', context=context)
print('답:', answer['answer'], f"(확신도 {answer['score']:.3f})")

## 4. 요약

긴 문서를 짧게 요약합니다.

In [ ]:
summarizer = pipeline('summarization', device=device)
article = '''Machine learning is a branch of artificial intelligence that focuses on
building systems that learn from data. Instead of being explicitly programmed,
these systems improve their performance as they are exposed to more data over time.
Deep learning, a subset of machine learning, uses neural networks with many layers
to model complex patterns in large datasets.'''
summary = summarizer(article, max_length=40, min_length=10, do_sample=False)
print('요약:', summary[0]['summary_text'])

## 5. 제로샷 분류

학습 없이, 우리가 제시한 후보 라벨 중에서 문장을 분류합니다.

In [ ]:
zero_shot = pipeline('zero-shot-classification', device=device)
result = zero_shot(
    'I want to refund the laptop I bought last week.',
    candidate_labels=['환불', '배송', '기술지원', '계정문의'])
for label, score in zip(result['labels'], result['scores']):
    print(f"{label}: {score:.3f}")

### 정리

- `pipeline` 하나로 감정분석·NER·질의응답·요약·제로샷 분류까지 학습 없이 수행했습니다.
- 빠른 프로토타이핑과 사전학습 모델 탐색에 유용합니다.
- 다음 노트북에서는 사전학습 모델을 우리 데이터로 **직접 파인튜닝**합니다.